# Lab Eval B — LLM-as-judge，然後**驗證你的 judge**（評保健品文案）

文案的「合規 / 切題 / 有沒有依據研究」**沒有標準答案** → 用模型當評審（LLM-as-judge）。
**但 judge 自己也會判錯——所以本關的高潮，是親手驗證「你的評審到底準不準」。**

敘事：**沒有標準答案 → 用 LLM-judge 打分 → 但 judge 會錯 → 一定要驗證（人工校準 + 換工具對照）。**

步驟：**① Vertex pointwise 打分 → ② Groundedness → ③ 換一家 RAGAS 對照 → ④ ⭐驗證 judge（本關高潮）→ ⑤ 找到盲點就調 rubric 重跑**。

> ④⑤ 這個「驗證 → 修正」的閉環，是我們盤點時**全公司專案都漏掉**的關鍵，也是你回到任何專案都能用的一招。
> 需要 GCP project（Vertex 有少量呼叫成本）；Colab 與你的 ADK 環境隔離，不會弄壞 lab1–5。

## 0. 安裝 + 認證（上傳 Service Account JSON）＋ 初始化

認證用 **Service Account JSON**（跟 Lab A 同一份）——公司政策擋「第三方 notebook 用**你的 Google 帳號**存取 GCP」，SA 是機器人身分、不走你帳號 OAuth → 繞過封鎖，走真的 Vertex。

In [ ]:
!pip install -q "google-cloud-aiplatform[evaluation]" ragas langchain-google-vertexai datasets

# 認證：上傳 Service Account JSON（跟 Lab A 同一份）
from google.colab import files
import os, json

print("請上傳 Service Account JSON 檔案（跟 Lab A 同一份）：")
uploaded = files.upload()
sa_filename = os.path.abspath(list(uploaded.keys())[0])
with open(sa_filename) as f:
    sa_info = json.load(f)

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = sa_filename
os.environ["GOOGLE_CLOUD_PROJECT"] = sa_info["project_id"]
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"   # Vertex Gen AI Eval 用區域

PROJECT = sa_info["project_id"]
LOCATION = "us-central1"

import vertexai
vertexai.init(project=PROJECT, location=LOCATION)
print("ready (SA):", PROJECT)

## 1. 準備評估資料(用保健品小組『產出的文案』)

為了自足,這裡直接放幾筆「agent 產出的文案 + 它依據的研究」。
- **`prompt`＝需求＋研究**（groundedness 只讀 prompt，所以研究要放進 prompt 才評得出「有沒有依據」）
- `response`＝產出文案 · `context`＝研究（給 RAGAS 用）
- 第 3 筆**故意寫壞**(宣稱療效＋誇大＋跟研究無關)當明顯反例；第 4 筆**踩線但不明顯**(宣稱改善睡眠、研究沒提)——是「驗證 judge」要抓的灰色地帶。

In [ ]:
import pandas as pd

# ⚠️ groundedness 評的是「response 有沒有依據 prompt 裡的資訊」→ 所以把研究放進 prompt，才評得出來
research = [
    "台灣魚油市場：需求上升，重視 Omega-3 / EPA / DHA；客群 30-45 上班族，重視專注與心血管。",
    "台灣益生菌市場：強調消化道機能；客群一般成人，重視日常保健。",
    "台灣魚油市場：需求上升，重視 Omega-3；客群 30-45 上班族。",
    "台灣魚油市場：需求上升，重視 Omega-3；客群 30-45 上班族。",
]

eval_dataset = pd.DataFrame({
    "prompt": [f"根據以下研究撰寫台灣市場行銷文案。研究：{r}" for r in research],
    "response": [
        "【純淨深海魚油】每日補充 Omega-3，幫助調節生理機能，守護上班族的專注日常。",
        "【日常益生菌】幫助維持消化道機能，早晚一次，陪你養成順暢好習慣。",
        "【神效魚油】全台銷量第一！立即見效、根治高血壓，無副作用保證有效！",  # 故意違規＋瞎編
        "【好眠魚油】幫助改善睡眠品質、放鬆助眠，每晚一顆一覺到天亮。",        # 踩線＋研究沒提睡眠
    ],
    "context": research,  # 保留一份給 RAGAS（跟研究相同）
})
eval_dataset

## 2. Vertex Gen AI Eval — pointwise(合規 + 切題,1–5 rubric)

自訂 `PointwiseMetric`:LLM 當評審,依 rubric 打 1/3/5 分。

In [ ]:
from vertexai.evaluation import EvalTask, PointwiseMetric, PointwiseMetricPromptTemplate

compliance_metric = PointwiseMetric(
    metric="compliance_relevance",
    metric_prompt_template=PointwiseMetricPromptTemplate(
        criteria={
            "合規性": "保健食品文案不得宣稱醫療療效(治療/根治/降血壓/預防疾病),不得誇大絕對化(第一/立即見效/無副作用/保證有效)。",
            "切題度": "文案要對應使用者指定的產品與市場。",
        },
        rating_rubric={
            "5": "完全合規且切題",
            "3": "部分踩線或偏題",
            "1": "嚴重違規(宣稱療效/誇大)或離題",
        },
        input_variables=["prompt"],
    ),
)

task = EvalTask(dataset=eval_dataset, metrics=[compliance_metric])
result = task.evaluate()
result.metrics_table[["prompt", "response", "compliance_relevance/score"]]

## 3. Groundedness(文案有沒有『依據研究』,不是瞎編)

用 Vertex 內建 groundedness 樣板:看 `response` 是否被 `context` 支持。

In [ ]:
from vertexai.evaluation import MetricPromptTemplateExamples

g_task = EvalTask(
    dataset=eval_dataset,
    metrics=[MetricPromptTemplateExamples.Pointwise.GROUNDEDNESS],
)
g_result = g_task.evaluate()
g_result.metrics_table[["response", "groundedness/score"]]
# groundedness 是二元：1=有依據 prompt 裡的研究、0=瞎編（研究沒說卻硬講）。
# 預期：第 1、2 筆（依據 Omega-3 / 消化道）→ 1；第 3 筆（根治高血壓）、第 4 筆（改善睡眠）研究都沒提 → 0。

## 4. 換一家：RAGAS faithfulness（對照，工具不獨大）｜⚠️ 加分項

同一件事（答案有沒有依據 context）用開源 RAGAS 再跑一次，對照兩邊——體現「工具不獨大」。
> ⚠️ RAGAS 對 langchain 版本很敏感，常有相依衝突。**跑不起來直接跳過**：本關核心是 pointwise ＋ groundedness ＋ 驗證 judge，RAGAS 只是加分對照。

In [ ]:
# 換一家：RAGAS faithfulness（加分對照）── 版本相依常出包，跑不起來就優雅跳過、不擋後面
try:
    from datasets import Dataset
    from ragas import evaluate
    from ragas.metrics import faithfulness
    from langchain_google_vertexai import ChatVertexAI, VertexAIEmbeddings

    ragas_ds = Dataset.from_dict({
        "question": eval_dataset["prompt"].tolist(),
        "answer": eval_dataset["response"].tolist(),
        "contexts": [[c] for c in eval_dataset["context"].tolist()],
    })
    ragas_result = evaluate(
        ragas_ds,
        metrics=[faithfulness],
        llm=ChatVertexAI(model_name="gemini-2.5-flash"),
        embeddings=VertexAIEmbeddings(model_name="text-embedding-004"),
    )
    print(ragas_result)
    print("對照：Vertex groundedness 與 RAGAS faithfulness 對『第 3 筆瞎編』應該都給低分。")
except Exception as e:
    print("⚠️ RAGAS 沒跑起來（常見的 langchain / ragas 版本相依問題）。")
    print("   這是『換一家工具對照』的加分項，跳過不影響核心（pointwise + groundedness + 驗證 judge）。")
    print("   錯誤：", type(e).__name__, "-", str(e)[:140])

## 5. ⭐ 驗證 judge：拿人工標註對照（本關高潮、全公司最常漏）

LLM-judge 自己也會判錯——**未經驗證的 judge，等於把主觀判斷丟給另一個模型還以為客觀了。**

做法：你先**當一次人工評審**替這幾題標分，再跟 judge 對照：
- 逐題看哪裡不一致；
- 算 **MAE、同意率、Cohen's κ**（κ＝校正掉「隨機也會猜中」的同意，投影片提到的更嚴謹版）；
- **重點是找出 judge 的盲點**——通常就是那個「踩線但不明顯」的灰色案例。

> 人工分（合規角度）：第 1、2 筆合規＝5；第 3 筆明顯違規＝1；第 4 筆「改善睡眠」踩線但不明顯＝3。

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score

# 你（人工）先標分——合規角度：5=合規, 3=踩線, 1=嚴重違規
human_labels = [5, 5, 1, 3]
judge_scores = result.metrics_table["compliance_relevance/score"].tolist()

# 逐題對照（最重要：看「哪一題」不一致，不是只看總分）
compare = pd.DataFrame({
    "response": eval_dataset["response"],
    "human": human_labels,
    "judge": judge_scores,
})
compare["一致?"] = np.where(
    (np.array(judge_scores) >= 4) == (np.array(human_labels) >= 4), "✓", "✗ 不一致"
)
print(compare.to_string(index=False))

mae = np.abs(np.array(human_labels, float) - np.array(judge_scores, float)).mean()
agree = np.mean((np.array(judge_scores) >= 4) == (np.array(human_labels) >= 4))

# Cohen's κ：校正掉「隨機也會猜中」的同意（投影片提到的更嚴謹版；1=完全一致、0=跟瞎猜一樣）
human_pass = (np.array(human_labels) >= 4).astype(int)
judge_pass = (np.array(judge_scores) >= 4).astype(int)
try:
    kappa = f"{cohen_kappa_score(human_pass, judge_pass):.2f}"
except Exception:
    kappa = "n/a（樣本太少/單一類別）"

print(f"\nMAE: {round(mae, 2)}（越小越貼近人工）")
print(f"同意率(>=4 合格): {agree:.0%}   ｜   Cohen's κ: {kappa}（此處 4 題僅示意，實務要更多題才穩）")
print("\n看哪一題標 ✗——通常是『踩線但不明顯』的第 4 筆。（judge 分數每次可能略有不同；重點是這個對照動作本身。）")

## 5b. 找到盲點 → 調 rubric 重跑（驗證的下一步：修正）

驗證不是「算完一致率」就結束。發現 judge 漏抓「改善睡眠」這種灰色地帶 → 把它**明確寫進 rubric**，再跑一次，看一致率有沒有上升。這就是實務上把 judge「養準」的循環。

> ⚠️ judge 是非確定的：若第一版 rubric 剛好就抓到了第 4 筆（沒有不一致），把第 4 筆換成更隱晦的踩線文案即可；重點是**「驗證 → 發現盲點 → 修 rubric → 再驗證」這個動作**，不是特定數字。

In [ ]:
# 發現 judge 漏抓「改善睡眠」→ 把這個灰色地帶明確寫進 rubric（v2），再跑一次
compliance_metric_v2 = PointwiseMetric(
    metric="compliance_relevance_v2",
    metric_prompt_template=PointwiseMetricPromptTemplate(
        criteria={
            "合規性": (
                "保健食品文案不得宣稱醫療療效或改善生理狀態，包括：治療/根治/降血壓/預防疾病，"
                "以及『改善睡眠/助眠/增強記憶/提升免疫』等受限宣稱；"
                "不得誇大絕對化(第一/立即見效/無副作用/保證有效)。"
            ),
            "切題度": "文案要對應使用者指定的產品與市場。",
        },
        rating_rubric={
            "5": "完全合規且切題",
            "3": "部分踩線（含改善睡眠等受限宣稱）或偏題",
            "1": "嚴重違規(宣稱療效/誇大)或離題",
        },
        input_variables=["prompt"],
    ),
)

result_v2 = EvalTask(dataset=eval_dataset, metrics=[compliance_metric_v2]).evaluate()
judge_v2 = result_v2.metrics_table["compliance_relevance_v2/score"].tolist()
agree_v2 = np.mean((np.array(judge_v2) >= 4) == (np.array(human_labels) >= 4))

print("rubric v1 judge:", judge_scores, f"→ 一致率 {agree:.0%}")
print("rubric v2 judge:", judge_v2, f"→ 一致率 {agree_v2:.0%}")
print("\n把灰色地帶寫進 rubric 後，judge 應更貼近人工——這就是『驗證 → 修正 → 再驗證』的閉環。")

## 收尾

- **LLM-as-judge** 能評沒有標準答案的品質/合規；
- **groundedness / faithfulness** 抓「瞎編」；
- **換一家工具(Vertex↔RAGAS)對照** → 不被單一工具綁架（sanity check）；
- ⭐ **拿人工驗證 judge → 找到盲點 → 調 rubric → 再驗證** → 這才是讓 judge 可信的閉環。

一句話：**judge 是工具，不是真理——要驗證、要修正，才可信。**
這也是你回到任何專案（不管用哪個工具）都能立刻套用的一招。